In [170]:
import pandas as pd
import numpy as np 

In [171]:
dataset1 = pd.read_csv('../data/used_cars_data.csv')
dataset2 = pd.read_csv('../data/used_car_dataset.csv')

Droping Unnessery column from the dataset 1

In [172]:
Drop_variables_dataset1 =['S.No.','Location','Engine','Power','Seats','New_Price','Mileage']
dataset1 = dataset1.drop(Drop_variables_dataset1, axis=1)

Spliting Name as model and brand


In [173]:
dataset1[['Brand','Model']] = dataset1['Name'].str.split(' ',n=1 ,expand = True)
dataset1 =dataset1.drop(['Name'],axis=1)
# rearranging the column 
cols = list(dataset1.columns)
cols.remove('Brand')
cols.remove('Model')
new_order = ['Brand', 'Model'] + cols
dataset1 = dataset1[new_order]

In [174]:
dataset1 = dataset1.iloc[:-1234, :]
#print(dataset1.isna().sum())
dataset1['Brand'] = dataset1['Brand'].replace('Land', 'Land Rover')
dataset1['Brand'] = dataset1['Brand'].replace('Maruti', 'maruti suzuki')

In [175]:
alphabetic_columns = ['Brand','Fuel_Type','Transmission','Owner_Type']
dataset1[alphabetic_columns] = dataset1[alphabetic_columns].map(lambda x: x.lower() if isinstance(x, str) else x)

In [176]:
dataset2 = dataset2.drop(['PostedDate','AdditionInfo','Age'],axis=1)

In [177]:
dataset2['AskPrice'] = dataset2['AskPrice'].str.replace('₹', '', regex=False)
dataset2['AskPrice'] = dataset2['AskPrice'].str.replace(',', '', regex=False)
dataset2['AskPrice'] = dataset2['AskPrice'].str.strip()
dataset2['AskPrice'] = pd.to_numeric(dataset2['AskPrice'], errors='coerce')
dataset1['Price'] = pd.to_numeric(dataset1['Price'],errors='coerce')
dataset2['AskPrice'] = dataset2['AskPrice'] / 100000  # convert to Lakhs
dataset2.rename(columns={'AskPrice': 'Price'}, inplace=True)

In [178]:
dataset2.rename(columns={
    'model': 'Model',
    'kmDriven': 'Kilometers_Driven',
    'FuelType': 'Fuel_Type',
    'Owner': 'Owner_Type'
}, inplace=True)
dataset2[alphabetic_columns] = dataset2[alphabetic_columns].map(lambda x: x.lower() if isinstance(x, str) else x)


In [179]:
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].str.replace(',', '', regex=False)
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].str.replace(' km', '', regex=False)
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].str.strip()
dataset2['Kilometers_Driven'] = pd.to_numeric(dataset2['Kilometers_Driven'], errors='coerce')

In [180]:
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].replace(0,pd.NA)

In [181]:
cols = ['Brand', 'Model', 'Year', 'Kilometers_Driven', 'Transmission', 'Owner_Type', 'Fuel_Type', 'Price']
dataset1_clean =dataset1[cols]
dataset2_clean =dataset2[cols]

dataset_merged = pd.concat([dataset1_clean , dataset2_clean], ignore_index=True)

In [182]:
# 1. Calculate your historical baseline metric once
median_km = dataset_merged['Kilometers_Driven'].median()
dataset_merged['Kilometers_Driven'] = dataset_merged['Kilometers_Driven'].fillna(median_km)

In [183]:
dataset_merged= dataset_merged[dataset_merged['Kilometers_Driven'] < 300000]
dataset_merged = dataset_merged[dataset_merged['Price'] < 100]
dataset_merged = dataset_merged.drop_duplicates()

In [184]:
# Keep only reasonable mileage records or true new cars
dataset_merged = dataset_merged[~((dataset_merged['Kilometers_Driven'] < 500) & (dataset_merged['Year'] < 2022))]

In [185]:
brand_counts = dataset_merged['Brand'].value_counts()
rare_brands = brand_counts[brand_counts < 10].index
dataset_merged['Brand'] = dataset_merged['Brand'].replace(rare_brands, 'other')

In [186]:
fuel_counts = dataset_merged['Fuel_Type'].value_counts()
rare_fuels = fuel_counts[fuel_counts < 21].index
dataset_merged['Fuel_Type'] = dataset_merged['Fuel_Type'].replace(rare_fuels, 'other')
print(dataset_merged['Owner_Type'].unique())


<StringArray>
['first', 'second', 'fourth & above', 'third']
Length: 4, dtype: str


In [187]:
from sklearn.preprocessing import OrdinalEncoder
o_encoder = OrdinalEncoder(categories=[['fourth & above', 'third', 'second', 'first']])
dataset_merged['Owner_Type'] =o_encoder.fit_transform(dataset_merged[['Owner_Type']])
print(dataset_merged['Owner_Type'].unique())

[3. 2. 0. 1.]


In [188]:

dataset_merged= dataset_merged.drop(['Model'],axis=1)

In [189]:
from sklearn.preprocessing import LabelEncoder 
le = LabelEncoder()
dataset_merged['Transmission'] = le.fit_transform(dataset_merged['Transmission']) 

In [190]:
dataset_merged= pd.get_dummies(dataset_merged, columns=['Brand', 'Fuel_Type'], drop_first=True).astype(int)


In [191]:
dataset_merged = dataset_merged[dataset_merged['Price'] >= 0.5]
print(dataset_merged.shape)

(14370, 37)


In [192]:
print(dataset_merged.shape)
dataset_merged.drop_duplicates(inplace=True)
print(dataset_merged.shape)

(14370, 37)
(14083, 37)


In [193]:
(dataset_merged ==0).sum()

Year                        0
Kilometers_Driven           0
Transmission             5591
Owner_Type                  9
Price                       0
Brand_bmw               13585
Brand_chevrolet         13898
Brand_datsun            14049
Brand_fiat              14035
Brand_ford              13565
Brand_honda             12801
Brand_hyundai           11683
Brand_isuzu             14071
Brand_jaguar            14014
Brand_jeep              14000
Brand_kia               13938
Brand_land rover        13966
Brand_lexus             14066
Brand_mahindra          13290
Brand_maruti suzuki     10573
Brand_mercedes-benz     13444
Brand_mg                14007
Brand_mini              14027
Brand_mitsubishi        14041
Brand_nissan            13923
Brand_other             14065
Brand_porsche           14052
Brand_renault           13729
Brand_skoda             13743
Brand_tata              13581
Brand_toyota            12980
Brand_volkswagen        13510
Brand_volvo             14019
Fuel_Type_

In [194]:
dataset_merged.to_csv('../data/cleaned_data.csv', index=False)